# Notebook 19 — MediaPipe Objectron: 3D Object Detection on CPU

**Part 7: Deep Learning for 6D Pose** | Estimated time: 45 min

---

> ⚠️ **Important Note:** MediaPipe Objectron was **deprecated** by Google in early 2023.  
> The API may not work with current MediaPipe versions (≥ 0.10).  
> This notebook teaches the **concepts and API structure** — they remain educational even if  
> Objectron itself is no longer actively maintained.  
> For production use, consider FoundationPose or FreeZe (NB 21).

### Why study it anyway?

Objectron demonstrated something important: **30+ FPS 3D pose estimation running entirely on CPU** — no GPU required. The architecture concepts (lightweight models, mobile inference, anchor-free detection) influenced all later approaches.

## Recommended Watch

> Watch this **before** opening the notebook — it shows Objectron running live, giving good visual context before the deprecated API notebook.

| Title | Link | Duration |
|---|---|---|
| **3D Object Detection with MediaPipe and OpenCV** | [▶ Watch](https://www.youtube.com/watch?v=f-Ibri14KMY) | ~15 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Note: mediapipe installation may vary by version
    !pip install mediapipe==0.9.3 opencv-python numpy matplotlib -q
    print('Running in Google Colab — packages installed')
    print('Note: Objectron may not be available in newer mediapipe versions')
else:
    print('Running locally — recommended: pip install mediapipe==0.9.3')
    print('Note: mediapipe Objectron was deprecated in 0.10+')

## Learning Objectives

By the end of this notebook you will be able to:

- Explain the Objectron model architecture and why it achieves real-time CPU performance
- Understand what `rotation`, `translation`, `landmarks_2d` outputs mean
- Write the complete Objectron inference pipeline
- Decode and visualize 3D bounding boxes from 2D landmarks
- Know the limitations of category-specific models vs. zero-shot approaches
- Simulate the pipeline without a live camera

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'OpenCV: {cv2.__version__}')

# Check MediaPipe availability
try:
    import mediapipe as mp
    print(f'MediaPipe: {mp.__version__}')
    MEDIAPIPE_AVAILABLE = True
    # Check if Objectron is available
    try:
        _ = mp.solutions.objectron
        OBJECTRON_AVAILABLE = True
        print('Objectron: AVAILABLE')
    except AttributeError:
        OBJECTRON_AVAILABLE = False
        print('Objectron: NOT AVAILABLE (deprecated in this MediaPipe version)')
        print('→ Notebook will run in simulation mode')
except ImportError:
    MEDIAPIPE_AVAILABLE = False
    OBJECTRON_AVAILABLE = False
    print('MediaPipe not installed.')
    print('Install with: pip install mediapipe==0.9.3')
    print('→ Notebook will run in simulation mode')

## 1. Objectron Architecture — How It Achieves CPU Real-Time

Objectron uses a two-stage pipeline designed for mobile/CPU deployment:

```
Stage 1: Object Detector (fast, lightweight)
  Input: Full image
  Output: 2D bounding boxes (which region contains the object)

Stage 2: 3D Landmark Predictor (runs only on cropped region)
  Input: Cropped image from Stage 1
  Output: 9 keypoints (center + 8 corners of 3D bounding box)
```

### The 9 3D Landmarks

```
3D bounding box corners (in object frame):

        4 ──────── 8
       /│          /│
      / │         / │
     2 ──────── 6  │
     │  3 ──────│── 7
     │ /        │ /
     │/         │/
     1 ──────── 5
         ★ 0 (center)

9 points × 2 coords (for 2D) = 18 values
```

### Why CPU is fast enough

- Stage 1 runs at full speed on the full image (few milliseconds with MobileNet)
- Stage 2 runs on a tiny crop (50×50–150×150 px) — tiny network, tiny input
- No convolutions on full 1280×720 frame needed at Stage 2
- Total: ~33ms per frame on a modern laptop CPU → ~30 FPS

In [ ]:
# Visualize the 3D bounding box landmarks structure
def draw_3d_box_landmarks():
    """Draw the 9-landmark 3D bounding box structure."""
    # 3D corners of a unit box
    corners_3d = np.array([
        [0, 0, 0],    # 0: center (not a corner)
        [-1, -1, -1], # 1
        [-1, -1,  1], # 2
        [-1,  1, -1], # 3
        [-1,  1,  1], # 4
        [ 1, -1, -1], # 5
        [ 1, -1,  1], # 6
        [ 1,  1, -1], # 7
        [ 1,  1,  1], # 8
    ], dtype=np.float32) * 0.5
    
    # Edges of the bounding box
    edges = [
        (1,2),(1,3),(1,5),(2,4),(2,6),(3,4),(3,7),
        (4,8),(5,6),(5,7),(6,8),(7,8)
    ]
    
    fig = plt.figure(figsize=(10, 5))
    
    ax = fig.add_subplot(121, projection='3d')
    for i, j in edges:
        p1, p2 = corners_3d[i], corners_3d[j]
        ax.plot([p1[0],p2[0]], [p1[1],p2[1]], [p1[2],p2[2]], 'b-', lw=1.5)
    ax.scatter(corners_3d[1:,0], corners_3d[1:,1], corners_3d[1:,2],
               c='blue', s=50, zorder=5)
    ax.scatter([corners_3d[0,0]], [corners_3d[0,1]], [corners_3d[0,2]],
               c='red', s=100, zorder=5, marker='*')
    for i, pt in enumerate(corners_3d):
        ax.text(pt[0], pt[1], pt[2], f' {i}', fontsize=9)
    ax.set_title('9 Landmarks in 3D\n(★=center, 1-8=corners)')
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    
    # Project to 2D (orthographic)
    ax2 = fig.add_subplot(122)
    proj_2d = corners_3d[:, :2] + np.array([0.2, 0.1]) * corners_3d[:, 2:3]
    for i, j in edges:
        p1, p2 = proj_2d[i], proj_2d[j]
        ax2.plot([p1[0],p2[0]], [p1[1],p2[1]], 'b-', lw=1.5)
    ax2.scatter(proj_2d[1:,0], proj_2d[1:,1], c='blue', s=50, zorder=5)
    ax2.scatter([proj_2d[0,0]], [proj_2d[0,1]], c='red', s=100, marker='*', zorder=5)
    for i, pt in enumerate(proj_2d):
        ax2.text(pt[0]+0.01, pt[1]+0.01, f'{i}', fontsize=9)
    ax2.set_xlim(-0.8, 0.8); ax2.set_ylim(-0.8, 0.8)
    ax2.set_aspect('equal')
    ax2.set_title('Projected 2D landmarks\n(what Objectron predicts in image space)')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlabel('u (pixels, normalized)'); ax2.set_ylabel('v (pixels, normalized)')
    
    plt.suptitle('MediaPipe Objectron: 9-point 3D bounding box representation',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

draw_3d_box_landmarks()

## 2. The Objectron API

The complete API (works with MediaPipe ≤ 0.9.3):

```python
import mediapipe as mp

mp_objectron = mp.solutions.objectron
mp_drawing   = mp.solutions.drawing_utils

with mp_objectron.Objectron(
    static_image_mode=False,          # False = video mode (uses tracking)
    max_num_objects=2,                 # max detections per frame
    min_detection_confidence=0.5,      # detection threshold
    min_tracking_confidence=0.99,      # tracking continuity
    model_name='Cup'                   # 'Cup', 'Shoe', 'Camera', 'Sneaker'
) as objectron:
    
    # For each frame:
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image_rgb.flags.writeable = False
    results = objectron.process(image_rgb)
    image_rgb.flags.writeable = True
    frame = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    if results.detected_objects:
        for detected_object in results.detected_objects:
            # Draw 2D box
            mp_drawing.draw_landmarks(
                frame,
                detected_object.landmarks_2d,
                mp_objectron.BOX_CONNECTIONS
            )
            # Draw 3D axes
            mp_drawing.draw_axis(
                frame,
                detected_object.rotation,
                detected_object.translation
            )
            # Access pose
            R = detected_object.rotation      # 3×3 rotation matrix
            t = detected_object.translation   # [X, Y, Z] translation
```

### Supported Object Categories

| Model Name | Example Use Case |
|---|---|
| `'Cup'` | Coffee cups, mugs, drinking glasses |
| `'Shoe'` | Athletic shoes, sneakers |
| `'Camera'` | DSLR cameras, camera bodies |
| `'Sneaker'` | Specific sneaker style |

⚠️ **These are NOT customizable.** Objectron only works on objects similar to its training data.

In [ ]:
# Complete Objectron pipeline (runs if mediapipe with Objectron is available)
if OBJECTRON_AVAILABLE:
    mp_objectron = mp.solutions.objectron
    mp_drawing   = mp.solutions.drawing_utils
    
    def run_objectron_on_image(image_bgr, model_name='Cup',
                               min_det=0.5, min_track=0.99):
        """
        Run Objectron on a single image.
        Returns annotated image and list of detected poses.
        """
        with mp_objectron.Objectron(
            static_image_mode=True,  # True for single images
            max_num_objects=3,
            min_detection_confidence=min_det,
            model_name=model_name
        ) as objectron:
            
            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            results = objectron.process(image_rgb)
            
            annotated = image_bgr.copy()
            poses = []
            
            if results.detected_objects:
                for obj in results.detected_objects:
                    mp_drawing.draw_landmarks(
                        annotated,
                        obj.landmarks_2d,
                        mp_objectron.BOX_CONNECTIONS
                    )
                    mp_drawing.draw_axis(annotated, obj.rotation, obj.translation)
                    poses.append({
                        'rotation': np.array(obj.rotation).reshape(3, 3),
                        'translation': np.array(obj.translation),
                    })
            
            return annotated, poses
    
    print('Objectron pipeline ready.')
    print('To test: call run_objectron_on_image(your_image_bgr, model_name="Cup")')
else:
    print('Objectron not available — running in simulation mode.')
    print('The code above shows the exact API — study the structure.')

## 3. Simulating Objectron Output

Since Objectron may not be available, we simulate what it would output and visualize it.  
This gives you the same visualization code you'd use with a real detection.

In [ ]:
def simulate_objectron_detection():
    """
    Simulate what Objectron's output would look like.
    Returns rotation matrix, translation vector, and 2D landmarks.
    """
    # Simulate a cup slightly tilted toward camera, at ~0.5m depth
    # Rotation: tilt 20° toward camera
    angle = np.radians(20)
    R_sim = np.array([
        [1,          0,           0],
        [0,  np.cos(angle), -np.sin(angle)],
        [0,  np.sin(angle),  np.cos(angle)]
    ])
    t_sim = np.array([0.05, -0.02, 0.50])  # 5cm right, 2cm up, 50cm depth
    
    # Simulate 3D bounding box corners (cup: 8cm diameter, 10cm tall)
    r, h = 0.04, 0.05  # radius, half-height
    corners_obj = np.array([
        [0, 0, 0],      # center
        [-r, -r, -h],   # corners
        [-r, -r,  h],
        [-r,  r, -h],
        [-r,  r,  h],
        [ r, -r, -h],
        [ r, -r,  h],
        [ r,  r, -h],
        [ r,  r,  h],
    ])
    
    # Transform to camera frame
    corners_cam = (R_sim @ corners_obj.T).T + t_sim
    
    # Project to image (simple pinhole, 640×480, fx=fy=500)
    fx, fy, cx, cy = 500, 500, 320, 240
    landmarks_2d = np.array([
        [fx * p[0]/p[2] + cx, fy * p[1]/p[2] + cy]
        for p in corners_cam
    ])
    
    return R_sim, t_sim, corners_obj, corners_cam, landmarks_2d

R_sim, t_sim, corners_obj, corners_cam, landmarks_2d = simulate_objectron_detection()

print('Simulated Objectron detection:')
print(f'  Rotation matrix R shape: {R_sim.shape}')
print(f'  Translation t: {t_sim} m')
print(f'  Distance: {np.linalg.norm(t_sim):.2f} m = {np.linalg.norm(t_sim)*100:.0f} cm')
print(f'  2D landmarks shape: {landmarks_2d.shape}')
print(f'  Landmarks (pixel coords):')
for i, pt in enumerate(landmarks_2d):
    label = 'center' if i == 0 else f'corner {i}'
    print(f'    [{i}] {label}: ({pt[0]:.0f}, {pt[1]:.0f})')

In [ ]:
def draw_3d_box_on_image(image, landmarks_2d, R, t, fx=500, fy=500, cx=320, cy=240):
    """
    Draw the 3D bounding box and coordinate axes on image.
    This is what mp_drawing.draw_landmarks + draw_axis does internally.
    """
    output = image.copy()
    pts = landmarks_2d.astype(int)
    
    # Box connections (same as mp_objectron.BOX_CONNECTIONS)
    edges = [
        (1,2),(1,3),(1,5),(2,4),(2,6),(3,4),(3,7),
        (4,8),(5,6),(5,7),(6,8),(7,8)
    ]
    
    for i, j in edges:
        p1, p2 = tuple(pts[i]), tuple(pts[j])
        cv2.line(output, p1, p2, (0, 200, 255), 2)
    
    # Draw landmarks
    cv2.circle(output, tuple(pts[0]), 6, (0, 0, 255), -1)  # center: red
    for i in range(1, 9):
        cv2.circle(output, tuple(pts[i]), 4, (255, 100, 0), -1)  # corners: orange
    
    # Draw coordinate axes (XYZ as RGB)
    center_cam = t
    axis_len = 0.05  # 5cm
    axes_world = np.array([[axis_len,0,0],[0,axis_len,0],[0,0,axis_len]])
    axes_cam = (R @ axes_world.T).T + center_cam
    
    def to_pixel(p):
        return (int(fx*p[0]/p[2]+cx), int(fy*p[1]/p[2]+cy))
    
    origin_px = to_pixel(center_cam)
    axis_colors = [(0,0,255), (0,255,0), (255,0,0)]  # X=red, Y=green, Z=blue
    axis_labels = ['X', 'Y', 'Z']
    for ax_pt, color, label in zip(axes_cam, axis_colors, axis_labels):
        tip_px = to_pixel(ax_pt)
        cv2.arrowedLine(output, origin_px, tip_px, color, 2, tipLength=0.3)
        cv2.putText(output, label, (tip_px[0]+5, tip_px[1]), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    # Annotation
    dist = np.linalg.norm(t)
    cv2.putText(output, f'Cup detected | depth={dist*100:.0f}cm',
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(output, f't=[{t[0]*100:+.0f},{t[1]*100:+.0f},{t[2]*100:.0f}]cm',
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,0), 1)
    
    return output

# Draw simulated detection
scene = np.ones((480, 640, 3), dtype=np.uint8) * 200
# Add some texture
cv2.rectangle(scene, (0, 300), (640, 480), (170, 170, 160), -1)  # floor
cv2.putText(scene, '[Simulation — real Objectron shows actual camera frame]',
            (30, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (100,100,100), 1)

annotated = draw_3d_box_on_image(scene, landmarks_2d, R_sim, t_sim)

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title('Simulated Objectron output — 3D bounding box + coordinate axes', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Real-Time Webcam Script

Complete webcam script — run locally with `mediapipe==0.9.3` installed.

In [ ]:
OBJECTRON_SCRIPT = '''
# objectron_live.py — requires mediapipe==0.9.3
# Usage: python objectron_live.py
# Press Q to quit, S to switch object class

import cv2
import mediapipe as mp
import numpy as np
import time

mp_objectron = mp.solutions.objectron
mp_drawing   = mp.solutions.drawing_utils

MODEL_NAMES  = ["Cup", "Shoe", "Camera", "Sneaker"]
model_idx    = 0

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

prev_time = time.time()

with mp_objectron.Objectron(
    static_image_mode=False,
    max_num_objects=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.99,
    model_name=MODEL_NAMES[model_idx]
) as objectron:
    
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break
        
        # Convert BGR→RGB for MediaPipe
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_rgb.flags.writeable = False
        
        results = objectron.process(image_rgb)
        
        image_rgb.flags.writeable = True
        image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
        
        if results.detected_objects:
            for i, obj in enumerate(results.detected_objects):
                # Draw 2D bounding box
                mp_drawing.draw_landmarks(
                    image,
                    obj.landmarks_2d,
                    mp_objectron.BOX_CONNECTIONS
                )
                # Draw 3D coordinate axes
                mp_drawing.draw_axis(image, obj.rotation, obj.translation)
                
                # Extract and display pose
                t = obj.translation
                dist = np.sqrt(t[0]**2 + t[1]**2 + t[2]**2) * 100  # cm
                cv2.putText(image, f"Obj {i}: dist={dist:.0f}cm",
                            (10, 60 + i*30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        else:
            cv2.putText(image, f"Searching for {MODEL_NAMES[model_idx]}...",
                        (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,150,255), 2)
        
        # FPS
        now = time.time()
        fps = 1.0 / (now - prev_time)
        prev_time = now
        cv2.putText(image, f"FPS: {fps:.0f} | Model: {MODEL_NAMES[model_idx]}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
        
        cv2.imshow("Objectron", image)
        key = cv2.waitKey(5) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("s"):
            model_idx = (model_idx + 1) % len(MODEL_NAMES)
            print(f"Switched to: {MODEL_NAMES[model_idx]}")

cap.release()
cv2.destroyAllWindows()
'''

print('Objectron live script (mediapipe==0.9.3 required):')
print('='*60)
print(OBJECTRON_SCRIPT)

## 5. Limitations and When to Use Objectron

### Limitations

| Limitation | Impact |
|---|---|
| Deprecated (0.10+) | Cannot use with current MediaPipe |
| Only 4 object categories | Cannot detect arbitrary objects |
| Single-object per category | Confuses multiple cups |
| Pose accuracy | ~±5° rotation, ~±2cm translation |
| Occlusion sensitivity | >40% occlusion → detection fails |

### When it WAS the right tool

| Use Case | Good fit? |
|---|---|
| AR demo with a coffee cup | ✅ Fast, no GPU, simple |
| Tracking a shoe for fitting app | ✅ Real-time CPU |
| Industrial robot picking custom parts | ❌ Wrong tool — use FreeZe |
| Research into DL pose estimation | ✅ Good reference architecture |

### What replaced it

- **FoundationPose** — any object + RGB-D, much more accurate
- **FreeZe** — any object + RGB-D, zero-shot, training-free
- **SAM-6D** — segment anything + 6D pose
- **MediaPipe's newer solutions** — hand, face, holistic tracking (still maintained)

## Recap

| Concept | Key Point |
|---|---|
| Objectron output | `rotation` (3×3 R), `translation` (3D t), `landmarks_2d` (9 points) |
| 9 landmarks | 1 center + 8 corners of 3D bounding box |
| CPU real-time | 2-stage pipeline: fast detector → tiny landmark predictor on crop |
| BGR→RGB | Required before MediaPipe, back after |
| `static_image_mode` | True for photos, False for video (uses tracking) |
| Deprecated | Use FoundationPose/FreeZe for new projects |

**Next:** EfficientPose — end-to-end 6D pose with a deeper DL model.

---
## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Extract pose from simulated detection
# ============================================================
# Using R_sim and t_sim from the simulation above:
# 1. Compute the distance from camera to object
# 2. Compute the Euler angles (roll, pitch, yaw) from R_sim
# 3. Convert R_sim to a Rodrigues vector using cv2.Rodrigues
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# dist = np.linalg.norm(t_sim)
# print(f'Distance: {dist*100:.1f}cm')
# 
# pitch = np.degrees(np.arctan2(-R_sim[2,0], np.sqrt(R_sim[0,0]**2 + R_sim[1,0]**2)))
# yaw   = np.degrees(np.arctan2(R_sim[1,0], R_sim[0,0]))
# roll  = np.degrees(np.arctan2(R_sim[2,1], R_sim[2,2]))
# print(f'Euler: roll={roll:.1f}° pitch={pitch:.1f}° yaw={yaw:.1f}°')
# 
# rvec, _ = cv2.Rodrigues(R_sim)
# print(f'Rodrigues: {rvec.flatten().round(4)} rad')
# print(f'Rotation angle: {np.degrees(np.linalg.norm(rvec)):.1f}°')

In [ ]:
# ============================================================
# EXERCISE 2: Draw a different object shape
# ============================================================
# Instead of a cube bounding box, draw a CYLINDER wireframe
# centered at t_sim with radius=0.04m, height=0.10m.
# Use 12 points on the top and bottom circle.
# Project to 2D and draw on the scene image.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# n = 12; r = 0.04; h = 0.05
# fx_, fy_, cx_, cy_ = 500, 500, 320, 240
# angles = np.linspace(0, 2*np.pi, n, endpoint=False)
# 
# top_pts    = np.column_stack([r*np.cos(angles), r*np.sin(angles), np.full(n, h)])
# bottom_pts = np.column_stack([r*np.cos(angles), r*np.sin(angles), np.full(n,-h)])
# 
# def proj(p, R, t):
#     pc = R @ p + t
#     return int(fx_*pc[0]/pc[2]+cx_), int(fy_*pc[1]/pc[2]+cy_)
# 
# cylinder_img = scene.copy()
# for i in range(n):
#     j = (i+1)%n
#     p1t = proj(top_pts[i],    R_sim, t_sim)
#     p2t = proj(top_pts[j],    R_sim, t_sim)
#     p1b = proj(bottom_pts[i], R_sim, t_sim)
#     p2b = proj(bottom_pts[j], R_sim, t_sim)
#     cv2.line(cylinder_img, p1t, p2t, (0,200,255), 2)
#     cv2.line(cylinder_img, p1b, p2b, (0,200,255), 2)
#     cv2.line(cylinder_img, p1t, p1b, (100,255,100), 1)
# 
# plt.figure(figsize=(8,6))
# plt.imshow(cv2.cvtColor(cylinder_img, cv2.COLOR_BGR2RGB))
# plt.title('Cylinder wireframe at object pose'); plt.axis('off'); plt.show()